In [0]:
import mlflow
from src.rag import RagConfig, CONFIG_FIELDS

user = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{user}/rag_experiments")

RUN_IDS = [
    "20261127T174908Z-1d4ad7",
    "20261127T175753Z-8be802",
    "20261127T180036Z-b48bd7",
    "20261127T180450Z-cddd14",
    "20261127T181143Z-de8f54",
]

cfg = RagConfig()
df = spark.table(f"{cfg.catalog}.{cfg.schema}.experiment_runs").toPandas()

for rid in RUN_IDS:
    rows = df[df.run_id == rid]
    first = rows.iloc[0]
    with mlflow.start_run(run_name=first["run_label"]):
        mlflow.log_params({c: first[c] for c in CONFIG_FIELDS if c in rows.columns})
        mlflow.log_params({"run_id": rid, "logged": "retroactively from experiment_runs"})
        mlflow.log_metric("n_questions", len(rows))
        if "error" in rows.columns:
            mlflow.log_metric("n_errors", int(rows["error"].notna().sum()))
    print(rid, first["variant"], len(rows))